# 🩺 ML-Based Early Disease Prediction System

**Dataset:** Blood samples with 24 biomarkers → Predicts: Diabetes, Heart Disease, Anemia, Thalassemia, Thrombocytopenia, or Healthy

**Pipeline:**
1. Data loading & EDA
2. Preprocessing & class balancing
3. Train multiple ML models
4. Evaluate & compare
5. Save best model
6. Build interactive prediction interface


## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required packages
!pip install imbalanced-learn xgboost lightgbm shap --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import shap
import joblib
import os

# Style
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#4CAF50','#F44336','#FF9800','#9C27B0','#00BCD4']

print('✅ All libraries loaded successfully!')

## 📂 Step 2: Upload & Load Dataset

In [ ]:
# Upload your CSV file
from google.colab import files
uploaded = files.upload()  # Upload: blood_samples_dataset_test.csv

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# Clean up truncated disease names
disease_map = {
    'Thalasse': 'Thalassemia',
    'Heart Di': 'Heart Disease',
    'Thromboc': 'Thrombocytopenia',
    'Diabetes': 'Diabetes',
    'Anemia':   'Anemia',
    'Healthy':  'Healthy'
}
df['Disease'] = df['Disease'].map(lambda x: disease_map.get(x.strip(), x.strip()))

print(f'Dataset shape: {df.shape}')
print(f'\nDisease distribution:')
print(df['Disease'].value_counts())
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('Basic Statistics:')
df.describe().round(3)

In [ ]:
# Disease distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['Disease'].value_counts()
axes[0].bar(counts.index, counts.values, color=PALETTE)
axes[0].set_title('Disease Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Disease')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=90)
axes[1].set_title('Disease Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('disease_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('⚠️  Note: Dataset is imbalanced — we will apply SMOTE to fix this.')

In [ ]:
# Feature distributions by disease
key_features = ['Glucose', 'HbA1c', 'Hemoglobin', 'Troponin',
                'C-reactive Protein', 'Platelets', 'BMI', 'Cholesterol']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for j, disease in enumerate(df['Disease'].unique()):
        subset = df[df['Disease'] == disease][feat]
        axes[i].hist(subset, alpha=0.5, label=disease, bins=20, color=PALETTE[j % len(PALETTE)])
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Normalized Value')
    if i == 0:
        axes[i].legend(fontsize=7)

plt.suptitle('Key Biomarker Distributions by Disease', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 11))
corr = df.drop('Disease', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## ⚙️ Step 4: Preprocessing & Train/Test Split

In [ ]:
# Encode target labels
le = LabelEncoder()
df['Disease_encoded'] = le.fit_transform(df['Disease'])

print('Label mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

X = df.drop(['Disease', 'Disease_encoded'], axis=1)
y = df['Disease_encoded']
feature_names = X.columns.tolist()

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print('Before SMOTE:', dict(zip(*np.unique(y_train, return_counts=True))))
print('After  SMOTE:', dict(zip(*np.unique(y_train_res, return_counts=True))))
print(f'\nClass names: {le.classes_}')

## 🤖 Step 5: Train & Compare Multiple ML Models

In [ ]:
# Define models
models = {
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors':    KNeighborsClassifier(n_neighbors=7),
    'Random Forest':          RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':      GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost':                xgb.XGBClassifier(n_estimators=200, use_label_encoder=False,
                                                 eval_metric='mlogloss', random_state=42),
    'LightGBM':               lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
    'SVM':                    SVC(kernel='rbf', probability=True, random_state=42)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Training models...\n')
for name, model in models.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_res, y_train_res, cv=cv,
                                 scoring='accuracy', n_jobs=-1)
    # Train on full resampled training set
    model.fit(X_train_res, y_train_res)
    # Test set evaluation
    y_pred = model.predict(X_test_scaled)
    test_acc = accuracy_score(y_test, y_pred)

    results[name] = {
        'CV Mean': cv_scores.mean(),
        'CV Std':  cv_scores.std(),
        'Test Acc': test_acc,
        'Model':   model
    }
    print(f'{name:<25} CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  |  Test Acc: {test_acc:.4f}')

print('\n✅ All models trained!')

In [ ]:
# Compare model performance visually
result_df = pd.DataFrame({
    'Model':    list(results.keys()),
    'CV Mean':  [v['CV Mean']  for v in results.values()],
    'CV Std':   [v['CV Std']   for v in results.values()],
    'Test Acc': [v['Test Acc'] for v in results.values()]
}).sort_values('Test Acc', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(result_df['Model'], result_df['Test Acc'], color=PALETTE * 2, alpha=0.85)
ax.errorbar(result_df['CV Mean'], result_df['Model'],
            xerr=result_df['CV Std'], fmt='D', color='black',
            capsize=5, label='CV Mean ± Std', markersize=6)
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('Model Comparison: Test Accuracy & Cross-Validation', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, result_df['Test Acc']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏆 Step 6: Evaluate Best Model

In [ ]:
# Select best model by test accuracy
best_name = max(results, key=lambda k: results[k]['Test Acc'])
best_model = results[best_name]['Model']
print(f'🏆 Best Model: {best_name} (Test Acc = {results[best_name]["Test Acc"]:.4f})')

# Detailed evaluation
y_pred_best = best_model.predict(X_test_scaled)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=feature_names)
    importances = importances.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 9))
    colors = ['#2196F3' if v < importances.quantile(0.75) else '#F44336'
              for v in importances.values]
    importances.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'Feature Importances — {best_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Top 5 most important features:')
    print(importances.tail(5)[::-1])
else:
    print(f'Feature importance not available for {best_name}. Skipping.')

## 🔮 Step 7: SHAP Explainability

In [ ]:
# SHAP values for model explainability
try:
    if hasattr(best_model, 'feature_importances_'):
        explainer = shap.TreeExplainer(best_model)
    else:
        explainer = shap.KernelExplainer(best_model.predict_proba,
                                          shap.sample(X_test_scaled, 50))

    shap_values = explainer.shap_values(X_test_scaled[:100])

    # Summary plot
    plt.figure(figsize=(10, 8))
    if isinstance(shap_values, list):
        shap.summary_plot(shap_values[0], X_test_scaled[:100],
                          feature_names=feature_names, show=False)
    else:
        shap.summary_plot(shap_values, X_test_scaled[:100],
                          feature_names=feature_names, show=False)
    plt.title('SHAP Feature Impact on Predictions', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ SHAP plot generated!')
except Exception as e:
    print(f'SHAP skipped: {e}')

## 🎯 Step 8: Build Ensemble (Voting Classifier)

In [ ]:
# Build a soft-voting ensemble from top 3 models
top3 = sorted(results.items(), key=lambda x: x[1]['Test Acc'], reverse=True)[:3]
print('Top 3 models for ensemble:')
for name, res in top3:
    print(f'  {name}: {res["Test Acc"]:.4f}')

ensemble = VotingClassifier(
    estimators=[(name.replace(' ', '_'), res['Model']) for name, res in top3],
    voting='soft'
)
ensemble.fit(X_train_res, y_train_res)
ensemble_acc = accuracy_score(y_test, ensemble.predict(X_test_scaled))
print(f'\n🎯 Ensemble Accuracy: {ensemble_acc:.4f}')

# Use best overall model for saving
final_model = ensemble if ensemble_acc > results[best_name]['Test Acc'] else best_model
final_acc   = max(ensemble_acc, results[best_name]['Test Acc'])
print(f'Final model accuracy: {final_acc:.4f}')

## 💾 Step 9: Save Model & Artifacts

In [ ]:
# Save all artifacts needed for deployment
os.makedirs('disease_prediction_model', exist_ok=True)

joblib.dump(final_model, 'disease_prediction_model/model.pkl')
joblib.dump(scaler,      'disease_prediction_model/scaler.pkl')
joblib.dump(le,          'disease_prediction_model/label_encoder.pkl')

# Save feature names
import json
with open('disease_prediction_model/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print('✅ Model artifacts saved:')
for f in os.listdir('disease_prediction_model'):
    print(f'   disease_prediction_model/{f}')

In [ ]:
# Zip and download the model folder
import shutil
shutil.make_archive('disease_prediction_model', 'zip', '.', 'disease_prediction_model')
files.download('disease_prediction_model.zip')
print('📦 Model package downloaded!')

## 🖥️ Step 10: Interactive Prediction Widget (Colab)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Disease info cards
DISEASE_INFO = {
    'Diabetes':         {'emoji': '🍬', 'color': '#FF9800',
                         'tip': 'Monitor blood sugar, maintain healthy weight, regular exercise.'},
    'Heart Disease':    {'emoji': '❤️', 'color': '#F44336',
                         'tip': 'Low-sodium diet, cardio exercise, regular BP monitoring.'},
    'Anemia':           {'emoji': '🩸', 'color': '#9C27B0',
                         'tip': 'Iron-rich diet, vitamin C supplementation, consult hematologist.'},
    'Thalassemia':      {'emoji': '🧬', 'color': '#3F51B5',
                         'tip': 'Regular blood transfusions may be needed. Consult genetic counselor.'},
    'Thrombocytopenia': {'emoji': '💊', 'color': '#00BCD4',
                         'tip': 'Avoid blood thinners, protect from injury, regular platelet checks.'},
    'Healthy':          {'emoji': '✅', 'color': '#4CAF50',
                         'tip': 'Keep up the healthy lifestyle! Annual check-ups recommended.'}
}

# Create sliders for each feature
sliders = {}
for feat in feature_names:
    sliders[feat] = widgets.FloatSlider(
        value=0.5, min=0.0, max=1.0, step=0.01,
        description='', layout=widgets.Layout(width='350px'),
        style={'description_width': 'initial'}
    )

output = widgets.Output()

def predict_disease(btn=None):
    with output:
        output.clear_output()
        values = np.array([[sliders[f].value for f in feature_names]])
        values_scaled = scaler.transform(values)
        pred_encoded = final_model.predict(values_scaled)[0]
        pred_proba   = final_model.predict_proba(values_scaled)[0]
        pred_disease = le.inverse_transform([pred_encoded])[0]
        confidence   = pred_proba.max() * 100
        info = DISEASE_INFO.get(pred_disease, {'emoji': '🔬', 'color': '#607D8B', 'tip': ''})

        # Results display
        print(f'\n{'='*55}')
        print(f'  {info["emoji"]}  PREDICTED CONDITION: {pred_disease.upper()}')
        print(f'  Confidence: {confidence:.1f}%')
        print(f'  💡 Tip: {info["tip"]}')
        print(f'{'='*55}')
        print('\nAll class probabilities:')
        for i, (cls, prob) in enumerate(zip(le.classes_, pred_proba)):
            bar = '█' * int(prob * 30)
            print(f'  {cls:<20} {bar:<30} {prob*100:.1f}%')

predict_btn = widgets.Button(
    description='🔍 Predict Disease',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
predict_btn.on_click(predict_disease)

# Arrange sliders in 2 columns
left_sliders  = []
right_sliders = []
for i, feat in enumerate(feature_names):
    row = widgets.HBox([
        widgets.Label(feat, layout=widgets.Layout(width='280px')),
        sliders[feat]
    ])
    if i % 2 == 0:
        left_sliders.append(row)
    else:
        right_sliders.append(row)

display(HTML('<h2 style="color:#1565C0">🩺 Early Disease Prediction System</h2>'))
display(HTML('<p style="color:gray">Adjust the biomarker values (normalized 0–1) and click Predict.</p>'))
display(widgets.HBox([
    widgets.VBox(left_sliders),
    widgets.VBox(right_sliders)
]))
display(predict_btn)
display(output)
print('🎛️  Widget ready! Adjust sliders and click Predict.')

## 📊 Step 11: Quick Single Prediction (Code-based)

In [ ]:
# Quick prediction function for code-based testing
def predict_from_dict(patient_values: dict) -> dict:
    """Predict disease from a dict of {feature_name: value}."""
    values = np.array([[patient_values.get(f, 0.5) for f in feature_names]])
    values_scaled = scaler.transform(values)
    pred_encoded = final_model.predict(values_scaled)[0]
    pred_proba   = final_model.predict_proba(values_scaled)[0]
    return {
        'prediction': le.inverse_transform([pred_encoded])[0],
        'confidence': float(pred_proba.max()),
        'probabilities': dict(zip(le.classes_, pred_proba.round(3)))
    }

# Example: test with a sample from the test set
sample_idx = 0
sample_row = X_test.iloc[sample_idx]
actual_disease = le.inverse_transform([y_test.iloc[sample_idx]])[0]

result = predict_from_dict(sample_row.to_dict())
print(f'Sample patient biomarkers (first 5 features):')
print(sample_row.head())
print(f'\n🎯 Actual disease:    {actual_disease}')
print(f'🤖 Predicted disease: {result["prediction"]}')
print(f'📊 Confidence:        {result["confidence"]*100:.1f}%')
print(f'\nAll probabilities:')
for disease, prob in sorted(result['probabilities'].items(), key=lambda x: -x[1]):
    print(f'  {disease:<20} {prob*100:.1f}%')

---
## ✅ Summary

| Step | Status |
|------|--------|
| Data Loading & EDA | ✅ |
| Class Imbalance Handling (SMOTE) | ✅ |
| 7 ML Models Trained & Compared | ✅ |
| Best Model Selected & Evaluated | ✅ |
| SHAP Explainability | ✅ |
| Ensemble Voting Classifier | ✅ |
| Model Saved & Downloaded | ✅ |
| Interactive Widget | ✅ |

**Next Step:** Deploy with Streamlit! See `streamlit_app.py` included in your package.
